In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Data Aggregation").getOrCreate()

26/05/05 13:50:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
reviews = spark.read.csv("reviews.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")

In [3]:
listings = spark.read.csv("listings.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")

In [5]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.functions import regexp_replace

listings = listings.withColumn('price_numeric', regexp_replace('price', '[$,]', '').cast('float'))

def categorize_price(price):
    if price is None:
        return 'Unknown'
    elif price < 50:
        return 'Budget'
    elif 50 <= price < 150:
        return 'mid-range'
    elif price >= 150:
        return 'luxury'
    else:
        return 'Unknown'

categorize_price_udf = udf(categorize_price, StringType())

listings_with_category = listings.filter(listings.price_numeric.isNotNull()).withColumn('price_category', categorize_price_udf(listings.price_numeric)).groupBy('price_category').count().show()

[Stage 4:>                                                          (0 + 1) / 1]

+--------------+-----+
|price_category|count|
+--------------+-----+
|        luxury|27516|
|        Budget| 6114|
|     mid-range|28333|
+--------------+-----+



In [6]:
from pyspark.sql.functions import avg
from pyspark.sql.types import FloatType

positive_words = {'good', 'great', 'excellent', 'amazing', 'fantastic', 'wonderful', 'pleasant', 'lovely', 'nice', 'enjoyed'}
negative_words = {'bad', 'terrible', 'awful', 'horrible', 'disappointing', 'poor', 'hate', 'unpleasant', 'dirty', 'noisy'}

def sentiment_score(comment):
    if comment is None:
        return 0.0
    comment_lower = comment.lower()
    score = 0

    for word in positive_words:
        if word in comment_lower:
            score += 1

    for word in negative_words:
        if word in comment_lower:
            score -= 1

sentiment_score_udf = udf(sentiment_score, FloatType())

reviews_with_sentiment = reviews.withColumn('sentiment_score', sentiment_score_udf(reviews.comments))

listings.join(reviews_with_sentiment, listings.id == reviews.listing_id, 'inner').groupBy('listing_id', 'name').agg(avg('sentiment_score').alias('average_sentiment')).orderBy('average_sentiment', ascending=False).select('listing_id', 'name', 'average_sentiment').show(truncate=False)

[Stage 8:>                                                          (0 + 1) / 1]

+----------+--------------------------------------------------+-----------------+
|listing_id|name                                              |average_sentiment|
+----------+--------------------------------------------------+-----------------+
|55402     |Modern 2 Bed 2 Bath, UK, Croydon                  |NULL             |
|362026    |Double room in lovely period house - friendly host|NULL             |
|810314    |BALCONY/OVERLOOKING RUGBY GROUND WI FI  WHOLE FLAT|NULL             |
|318287    |safe and spacious room in comfy family home       |NULL             |
|1038126   |Crayford - Camden Islington                       |NULL             |
|436257    |Sunny double room/balcony/shower/wc               |NULL             |
|991093    |Cozy Flat Just off Upper Street, N1               |NULL             |
|410971    |Beautiful Penthouse in Clapham                    |NULL             |
|1208955   |Relaxing Georgian house in London                 |NULL             |
|469676    |Doub

In [7]:
reviews.createOrReplaceTempView("reviews")
listings.createOrReplaceTempView("listings")

sql_query = """
select r.listing_id, avg(length(r.comments)) as average_comment_length, count(r.id) as reviews_count
from reviews r join listings l on r.listing_id = l.id
group by r.listing_id
having count(r.id) >= 5
order by average_comment_length desc
"""

spark.sql(sql_query).show()

26/05/05 14:03:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 12:>                                                         (0 + 1) / 1]

+------------------+----------------------+-------------+
|        listing_id|average_comment_length|reviews_count|
+------------------+----------------------+-------------+
|618608352812465378|    1300.1666666666667|            6|
|          28508447|    1089.3333333333333|            6|
|          22661311|     1035.857142857143|            7|
|          53145228|    1006.6666666666666|            6|
|627425975703032358|     951.7777777777778|            9|
|           2197681|                 939.2|            5|
|          13891813|                 905.0|            5|
|            979753|     893.9230769230769|           13|
|630150178279666225|     890.7272727272727|           11|
|           8856894|     890.1666666666666|            6|
|          33310686|     885.8333333333334|            6|
|          22524075|                 885.0|            5|
|          29469389|                 885.0|            6|
|           5555679|     878.7169811320755|          106|
|           65

In [11]:
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import PandasUDFType
import pandas as pd

@pandas_udf(DoubleType(), functionType=PandasUDFType.GROUPED_AGG)
def average_days_since_first_review_udf(first_review_series) -> float:
    today = pd.to_datetime('today')
    listing_ages = (today - pd.to_datetime(first_review_series)).dt.days
    if listing_ages.empty:
        return None
    return listing_ages.mean()

listings.filter(listings.first_review.isNotNull()).groupBy('host_id').agg(average_days_since_first_review_udf(listings.first_review).alias('average_days_since_first_review_days')).show()

[Stage 20:>                                                         (0 + 1) / 1]

+-------+------------------------------------+
|host_id|average_days_since_first_review_days|
+-------+------------------------------------+
|   6774|                  2284.1666666666665|
|   9089|                               767.0|
|   9323|                              3277.0|
|  10657|                              1713.5|
|  11333|                               925.0|
|  11431|                              3873.0|
|  14596|                              2960.0|
|  19195|                              3924.0|
|  25235|                               257.0|
|  26258|                               332.0|
|  30577|                              1251.0|
|  30780|                              3561.0|
|  32851|                              3794.5|
|  34007|                               629.0|
|  36808|                               662.0|
|  38691|                              1258.0|
|  40515|                              2520.0|
|  40944|                   736.9285714285714|
|  41759|    

Traceback (most recent call last):                                              
  File "/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
26/05/05 14:09:50 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /private/var/folders/np/6gy7xrp56fz4y49y54thj8xw0000gn/T/blockmgr-ce0f0cbc-a267-4e80-a025-81a9b88132d1. Falling back to Java IO way
java.io.IOException: Failed to delete: /private/var/folders/np/6gy7xrp56fz4y49y54thj8xw0000gn/T/blockmgr-ce0f0cbc-a267-4e80-a025-81a9b88132d1
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.j